# Model Training - CNN 1D Architecture

- **Authored by:** Matheus Ferreira Silva 
- **GitHub:**: https://github.com/MatheusFS-dev

## 1. Setup and Configuration

### 1.1. Environment Variables

In [1]:
import os

# Specify GPU to use (e.g., GPU:0, CPU:-1)
# os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# Allow TensorFlow to allocate GPU memory as needed
os.environ['TF_FORCE_GPU_ALLOW_GROWTH'] = 'true'

# If it fails to determine best cudnn convolution algorithm
os.environ["XLA_FLAGS"] = "--xla_gpu_strict_conv_algorithm_picker=false"

In [2]:
# Disable all auto-JIT clustering at the process level
os.environ["TF_XLA_FLAGS"] = "--tf_xla_auto_jit=-1"

### 1.2. Imports

In [3]:
from _imports import * # Centralized file containing all imports

2025-08-14 15:12:50.873136: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-08-14 15:12:50.886899: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1755195170.903115  446121 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1755195170.908008  446121 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-08-14 15:12:50.924707: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

### 1.3. GPU Management

In [4]:
get_gpu_info()


TensorFlow GPU Monitor - 2025-08-14 15:12:52
TensorFlow Configuration
Version        : 2.18.0
CUDA Support   : Yes
CUDA Version   : 12.5.1
cuDNN Version  : 9

GPU Information
GPU Name                      Memory Usage         Temp   Util  
--------------------------------------------------------------------------------
0   NVIDIA GeForce RTX 3070      0.9GB /    8.0GB  49C    29%   



2025-08-14 15:12:52.794924: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1755195172.795891  446121 gpu_device.cc:2022] Created device /device:GPU:0 with 5371 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3070 Ti, pci bus id: 0000:b3:00.0, compute capability: 8.6


## 2. Run Parameters 

In [5]:
EPOCHS = 100
BATCH_SIZE = 64

DATA_SEED = 99
TRAIN_SEED = 333

# Set Python, NumPy, Keras and TensorFlow seeds
set_random_seed(TRAIN_SEED)

# Reproducibility settings for TensorFlow:
# Note: must have same inputs and hardware
# Warning: this affects overall performance
# tf.config.experimental.enable_op_determinism()

# Enable or disable XLA compilation
# Note: some layers don't support determinism with XLA
JIT_COMPILE = False

In [6]:
POLICY = mixed_precision.Policy("mixed_float16")
#! Tried using mixed_float16, but got overflow to inf and propagate nan
mixed_precision.set_global_policy(POLICY)

BYTES_PER_PARAM = tf.dtypes.as_dtype(POLICY.variable_dtype).size

In [7]:
# Set to an existing dir to resume training
RUN_DIR = f"runs/{get_caller_stem()}"  # (e.g. "runs/train_1")

## 3. Data Loading and Preprocessing

In [8]:
(
    s008_coord_input,
    s008_lidar_input,
    s008_y_train,
    s009_coord_input,
    s009_lidar_input,
    s009_y,
) = load_dataset_sparse_labels()

/home/matheus/src/RayWise/src/_load_dataset.py:42: ComplexWarning: Casting complex values to real discards the imaginary part
  s008_y_train = s008_y_train.astype(np.float32)
/home/matheus/src/RayWise/src/_load_dataset.py:65: ComplexWarning: Casting complex values to real discards the imaginary part
  s008_y_val = s008_y_val.astype(np.float32)


Shape before conversion: (9234, 8, 32)
Shape after conversion: (9234,)
y_train shape: (9234,)
coord_input shape: (9234, 2)
lidar_input shape: (9234, 20, 200, 10)
Shape before conversion: (1960, 8, 32)
Shape after conversion: (1960,)
y_val shape: (1960,)
coord_input_val shape: (1960, 2)
lidar_input_val shape: (1960, 20, 200, 10)
y_train shape: (11194,)
coord_input shape: (11194, 2)
lidar_input shape: (11194, 20, 200, 10)
Shape before conversion: (9638, 8, 32)
Shape after conversion: (9638,)
y shape: (9638,)
coord_input shape: (9638, 2)
lidar_input shape: (9638, 20, 200, 10)


/home/matheus/src/RayWise/src/_load_dataset.py:101: ComplexWarning: Casting complex values to real discards the imaginary part
  s009_y = s009_y.astype(np.float32)


In [9]:
(
    x_s008_lidar_train,
    x_s008_lidar_val,
    x_s008_coord_train,
    x_s008_coord_val,
    y_s008_train,
    y_val,
) = train_test_split(
    s008_lidar_input,
    s008_coord_input,
    s008_y_train,
    test_size=0.2,
    random_state=DATA_SEED,
    shuffle=True,
)

(
    x_s009_lidar_test,
    x_s009_lidar_val,
    x_s009_coord_test,
    x_s009_coord_val,
    y_s009_test,
    y_s009_val,
) = train_test_split(
    s009_lidar_input,
    s009_coord_input,
    s009_y,
    test_size=0.2,
    random_state=DATA_SEED,
    shuffle=True,
)

x_lidar_train = x_s008_lidar_train
x_coord_train = x_s008_coord_train
y_train = y_s008_train

x_lidar_val = np.concatenate((x_s008_lidar_val, x_s009_lidar_val), axis=0)
x_coord_val = np.concatenate((x_s008_coord_val, x_s009_coord_val), axis=0)
y_val = np.concatenate((y_val, y_s009_val), axis=0)

x_lidar_test = x_s009_lidar_test
x_coord_test = x_s009_coord_test
y_test = y_s009_test

## 4. Model Definition

In [10]:
def build_model(show_summary: bool = True) -> Model:
    # ———————————————————————————————————————————————————————————————————————————— #
    #                              Model Construction                              #
    # ———————————————————————————————————————————————————————————————————————————— #

    # ———————————————————————————————— LiDAR Input ——————————————————————————————— #
    x_lidar_input = layers.Input(shape=(20, 200, 10), name="lidar_input")

    # Inline one-hot encoding of semantic values
    one_hot_lidar = layers.Lambda(
        lambda x: tf.concat(
            [
                # “Is there a BS anywhere in the 10 channels?” → 1 channel
                tf.cast(tf.reduce_any(tf.equal(x, -2), axis=-1, keepdims=True), tf.float32),
                # “Vehicle?” → 1 channel
                tf.cast(tf.reduce_any(tf.equal(x, -1), axis=-1, keepdims=True), tf.float32),
                # “Obstacle?” → 1 channel
                tf.cast(tf.reduce_any(tf.equal(x, 1), axis=-1, keepdims=True), tf.float32),
                # “Free?” → 1 channel (all channels zero)
                tf.cast(tf.reduce_all(tf.equal(x, 0), axis=-1, keepdims=True), tf.float32),
            ],
            axis=-1,
        ),
        #! Lambda has deserialization issues, so providing the output shape is necessary
        output_shape=(20, 200, 4),
        name="lidar_transform_to_one_hot",
    )(x_lidar_input)
    # -> (batch, 20, 200, 4)

    # Flatten the 20×200 grid into a 4000-length sequence with the 4 channels
    x_lidar_flat: layers.Layer = layers.Reshape((20 * 200, 4), name="lidar_flatten_4_channels")(one_hot_lidar)

    # ———————————————————————————————— GPS Input ———————————————————————————————— #
    # Input for coordinate data (e.g., shape: (2,))
    x_coord_input = layers.Input(shape=(2,), name="coord_input")

    # Turn (batch,2) → (batch,1,2) → tile to (batch,4000,2)'
    x_coord: layers.Layer = layers.Lambda(
        lambda x: tf.tile(tf.expand_dims(x, axis=1), [1, 20 * 200, 1]),
        #! Lambda has deserialization issues, so providing the output shape is necessary
        output_shape=(20 * 200, 2),
        name="coord_tile_flat",
    )(x_coord_input)

    # ————————————————————————————— Combine Branches ————————————————————————————— #
    # Fuse channels:  (batch,4000,4) + (batch,4000,2) → (batch,4000,6)
    combined = layers.Concatenate(axis=-1, name="combine_lidar_coord")([x_lidar_flat, x_coord])

    # ———————————————————————————————— Initializer ———————————————————————————————— #
    initializer = tf.keras.initializers.GlorotUniform(
        seed=TRAIN_SEED,
    )

    # ———————————————————————————————————— GNN ——————————————————————————————————— #
    from spektral.layers import GraphMasking, GlobalAvgPool, ChebConv
    A = build_knn_adjacency(rows=20, cols=200, k=12)

    x_graph, a_graph = GraphMasking()([combined, A])

    gnn1 = ChebConv(
        channels=380,
        K=2,
        activation=None,
        name="cheb_1",
    )([x_graph, a_graph])
    gnn1 = layers.Activation("relu", name="cheb_act_1")(gnn1)
    gnn1 = layers.Dropout(0.25, name="cheb_drop_1")(gnn1)

    gnn2 = ChebConv(
        channels=370,
        K=2,
        activation=None,
        name="cheb_2",
    )([gnn1, a_graph])
    # This layer has no activation
    gnn2 = layers.Dropout(0.05, name="cheb_drop_2")(gnn2)

    gnn3 = ChebConv(
        channels=270,
        K=2,
        activation=None,
        name="cheb_3",
    )([gnn2, a_graph])
    gnn3 = layers.Activation("elu", name="cheb_act_3")(gnn3)
    gnn3 = layers.Dropout(0.5, name="cheb_drop_3")(gnn3)

    # skip: gnn1 -> gnn2 (concat, resize-safe)
    _gnn12 = resize_for_skip_1d(gnn1, gnn2.shape[1], name="skip_gnn1_to_gnn2_resize")
    gnn2 = layers.Concatenate(axis=-1, name="skip_from_gnn1_to_gnn2")([_gnn12, gnn2])

    # skip: gnn2 -> gnn3 (concat)
    _gnn23 = resize_for_skip_1d(gnn2, gnn3.shape[1], name="skip_gnn2_to_gnn3_resize")
    gnn3 = layers.Concatenate(axis=-1, name="skip_from_gnn2_to_gnn3")([_gnn23, gnn3])

    # ———————————————————————————————————— DNN ——————————————————————————————————— #
    # Global pooling layer to reduce the graph to a fixed-size vector
    #! The GlobalAvgPool layer from Spektral causes an error with the skip connection,
    #! it adds a singleton channel dimension to 2d tensors. Probably due to the masking.
    x = gnn3
    x = GlobalAvgPool(name="global_avg_pool")(x)

    warnings.filterwarnings("ignore", message=".*Flatten.*mask.*support masking.*")

    # Flatten to remove the singleton dimension
    x = layers.Flatten(name="flatten_gnn_output")(x)

    dnn1 = layers.Dense(units=250, activation=None, kernel_initializer=initializer, name="dense_1")(x)
    dnn1 = layers.Activation("elu", name="dense_act_1")(dnn1)

    dnn2 = layers.Dense(units=550, activation=None, kernel_initializer=initializer, name="dense_2")(dnn1)
    dnn2 = layers.Activation("tanh", name="dense_act_2")(dnn2)
    dnn2 = layers.Concatenate(axis=-1, name="skip_from_dnn1_to_dnn2")([dnn1, dnn2])

    dnn3 = layers.Dense(units=600, activation=None, kernel_initializer=initializer, name="dense_3")(dnn2)
    dnn3 = layers.Activation("tanh", name="dense_act_3")(dnn3)
    dnn3 = layers.Dropout(rate=0.2)(dnn3)
    dnn3 = layers.Concatenate(axis=-1, name="skip_from_dnn2_to_dnn3")([dnn2, dnn3])

    dnn4 = layers.Dense(units=300, activation=None, kernel_initializer=initializer, name="dense_4")(dnn3)
    dnn4 = layers.Activation("elu", name="dense_act_4")(dnn4)
    dnn4 = layers.Dropout(rate=0.4)(dnn4)

    # —————————————————————————————————— Output —————————————————————————————————— #
    x = dnn4
    outputs = layers.Dense(
        256,
        activation="softmax",
        name="output",
        kernel_initializer=initializer,
    )(x)

    # —————————————————————————— Set Inputs and Outputs —————————————————————————— #
    model = Model(inputs=(x_lidar_input, x_coord_input), outputs=(outputs,))

    # ———————————————————————————————— Compilation ——————————————————————————————— #
    model.summary() if show_summary else None

    # steps_per_epoch = max(1, len(y_train) // BATCH_SIZE)
    # total_steps = steps_per_epoch * EPOCHS
    # warmup_steps = max(10, int(0.05 * total_steps))
    # decay_steps = max(1, total_steps - warmup_steps)

    # lr_schedule = optimizers.schedules.CosineDecay(
    #     initial_learning_rate=0.0,
    #     decay_steps=decay_steps,
    #     alpha=0.0,
    #     warmup_target=7e-5,
    #     warmup_steps=warmup_steps,
    # )

    # optimizer = optimizers.Lion(learning_rate=lr_schedule)

    optimizer = optimizers.Lion(learning_rate=7e-5, clipnorm=1.0)

    model.compile(
        optimizer=optimizer,
        loss=losses.SparseCategoricalCrossentropy(),
        metrics=["accuracy"],
        jit_compile=JIT_COMPILE,
    )

    return model

## Main

In [ ]:
try:
    # ——————————————————————————————————— Setup —————————————————————————————————— #
    (
        study_dir,
        args_dir,
        fig_dir,
        backup_dir,
        history_dir,
        model_dir,
        logs_dir,
        tensorboard_dir,
    ) = init_study_dirs(RUN_DIR, study_name="model_training")

    # ——————————————————————————————— Prepare Data ——————————————————————————————— #
    coord_scaler = StandardScaler()

    coord_scaler.fit(x_coord_train)
    x_coord_train = coord_scaler.transform(x_coord_train)
    x_coord_val = coord_scaler.transform(x_coord_val)
    x_coord_test = coord_scaler.transform(x_coord_test)
    s009_coord_input = coord_scaler.transform(s009_coord_input)

    # —————————————————————————————— Train the Model ————————————————————————————— #

    model = build_model(show_summary=True)
    save_model_plot(model, output_path=os.path.join(fig_dir, "model_plot.png"))

    history = model.fit(
        x=[x_lidar_train, x_coord_train],
        y=y_train,
        validation_data=([x_lidar_val, x_coord_val], y_val),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=get_callbacks_model(
            backup_dir=os.path.join(backup_dir, "training"),
            checkpoint_dir=os.path.join(backup_dir, "checkpoints"),
            #! Can cause high memory usage
            # tensorboard_logs=tensorboard_dir,
            early_stopping_patience=None,
            reduce_lr_patience=None,
        ),
        verbose=2,
    )

    model.save(os.path.join(model_dir, "model.keras"))

    # ——————————————————————————————— Test on S009 ——————————————————————————————— #
    test_loss, test_acc = model.evaluate(
        [x_lidar_test, x_coord_test], y_test, batch_size=BATCH_SIZE, verbose=0
    )
    # Now evaluate on the full s009 dataset for comparison purposes
    test_loss_full, test_acc_full = model.evaluate(
        [s009_lidar_input, s009_coord_input], s009_y, batch_size=BATCH_SIZE, verbose=0
    )

    # ——————————————————————————————— Save history ——————————————————————————————— #
    history_path = os.path.join(history_dir, "history.csv")
    history_data = {
        "epoch": list(range(1, len(history.history["loss"]) + 1)),
        "train_loss": history.history["loss"],
        "val_loss": history.history["val_loss"],
        "test_loss_s009": test_loss,
        "test_acc_s009": test_acc,
        "test_loss_s009_full": test_loss_full,
        "test_acc_s009_full": test_acc_full,
    }

    history_df = pd.DataFrame(history_data)
    history_df.to_csv(history_path, index=False)

    # ———————————————————————————————— Model Stats ——————————————————————————————— #
    write_model_stats_to_file(
        model=model,
        file_path=os.path.join(args_dir, "model_stats.txt"),
        batch_size=BATCH_SIZE,
        bytes_per_param=tf.dtypes.as_dtype(POLICY.variable_dtype).size,
        device=0,
        n_trials=1000,
        verbose=True,
        extra_attrs={
            "final_loss": history.history["loss"][-1],
            "final_val_loss": history.history["val_loss"][-1],
            "test_loss_s009": test_loss,
            "test_acc_s009": test_acc,
            "test_loss_s009_full": test_loss_full,
            "test_acc_s009_full": test_acc_full,
        },
    )
    # ———————————————————————————————————————————————————————————————————————————— #
except Exception as e:
    print(f"\n An error occurred: {e}\n")
    traceback.print_exc()

    with open(os.path.join(logs_dir, "training_error.log"), "a") as f:
        f.write(f"An error occurred during training:\n{e}\n{traceback.format_exc()}\n\n")
finally:
    # Clean up directories
    shutil.rmtree(backup_dir, ignore_errors=True)
    if not os.listdir(logs_dir):
        os.rmdir(logs_dir)

I0000 00:00:1755195174.137132  446121 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 5371 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3070 Ti, pci bus id: 0000:b3:00.0, compute capability: 8.6
2025-08-14 15:12:54.169226: E tensorflow/core/util/util.cc:131] oneDNN supports DT_HALF only on platforms with AVX-512. Falling back to the default Eigen-based implementation if present.
/home/matheus/anaconda3/envs/tf-optuna-araras/lib/python3.11/site-packages/keras/src/layers/layer.py:932: UserWarning: Layer 'skip_gnn1_to_gnn2_resize' (of type Lambda) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(
/home/matheus/anaconda3/envs/tf-optuna-araras/lib/python3.11/site-packages/keras/src/layers/layer.py:932: UserWarning: Layer 'skip_gnn2_to_gnn3_resize' (of type Lambda) was passed an input with a mask attac

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ lidar_input         │ (None, 20, 200,   │          0 │ -                 │
│ (InputLayer)        │ 10)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cast (Cast)         │ (None, 20, 200,   │          0 │ lidar_input[0][0] │
│                     │ 10)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ coord_input         │ (None, 2)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lidar_transform_to… │ (None, 20, 200,   │          0 │ cast[0][0]        │
│ (Lambda)            │ 4)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cast_1 (Cast)       │ (None, 2)         │          0 │ coord_input[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lidar_flatten_4_ch… │ (None, 4000, 4)   │          0 │ lidar_transform_… │
│ (Reshape)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ coord_tile_flat     │ (None, 4000, 2)   │          0 │ cast_1[0][0]      │
│ (Lambda)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ combine_lidar_coord │ (None, 4000, 6)   │          0 │ lidar_flatten_4_… │
│ (Concatenate)       │                   │            │ coord_tile_flat[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ graph_masking       │ [(None, 4000, 5), │          0 │ combine_lidar_co… │
│ (GraphMasking)      │ (4000, 4000)]     │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item (GetItem)  │ (None, 4000, 1)   │          0 │ combine_lidar_co… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cheb_1 (ChebConv)   │ (None, 4000, 380) │      4,180 │ graph_masking[0]… │
│                     │                   │            │ graph_masking[0]… │
│                     │                   │            │ get_item[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cheb_act_1          │ (None, 4000, 380) │          0 │ cheb_1[0][0]      │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cheb_drop_1         │ (None, 4000, 380) │          0 │ cheb_act_1[0][0]  │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ skip_gnn1_to_gnn2_… │ (None, 4000, 380) │          0 │ cheb_drop_1[0][0… │
│ (Lambda)            │                   │            │ get_item[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cheb_2 (ChebConv)   │ (None, 4000, 370) │    281,570 │ cheb_drop_1[0][0… │
│                     │                   │            │ graph_masking[0]… │
│                     │                   │            │ get_item[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cheb_drop_2         │ (None, 4000, 370) │          0 │ cheb_2[0][0]      │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ ones_like           │ (None, 4000, 380) │          0 │ skip_gnn1_to_gnn

 Total params: 1,857,076 (7.08 MB)

 Trainable params: 1,857,076 (7.08 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/100


/home/matheus/anaconda3/envs/tf-optuna-araras/lib/python3.11/site-packages/keras/src/layers/layer.py:932: UserWarning: Layer 'skip_gnn1_to_gnn2_resize' (of type Lambda) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(
/home/matheus/anaconda3/envs/tf-optuna-araras/lib/python3.11/site-packages/keras/src/layers/layer.py:932: UserWarning: Layer 'skip_gnn2_to_gnn3_resize' (of type Lambda) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(
